# Comparative Analysis of CNN and Efficient Vision Transformers for Gleason Grade Classification

**Course:** CSELEC2C Machine Learning (2025–2026)  
**Models:** ResNet-50 · TinyViT-11M · FastViT-T8  
**Dataset:** SICAPv2 (18,783 patches, 4-class: NC / G3 / G4 / G5)

---
This notebook runs end-to-end: dataset check → training → evaluation → Grad-CAM → results table.

## 0. Environment Setup

In [ ]:
# Install dependencies (Colab)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'timm==1.0.3', 'PyYAML', 'openpyxl', 'matplotlib', 'pandas'])

In [ ]:
# Mount Google Drive if running in Colab (optional — for checkpoint saving)
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    print('Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('Not running in Colab — Drive not mounted.')

In [ ]:
import os
from pathlib import Path

# ── Project root resolution ──────────────────────────────────────────────────
# The notebook lives at <project_root>/notebooks/main.ipynb.
# Resolve one level up so all relative paths stay consistent.
NOTEBOOK_DIR = Path(os.path.abspath('__file__')).parent if '__file__' in dir() else Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # <project_root>/

# Allow manual override (e.g. when cloning to a custom Drive path)
# PROJECT_ROOT = Path('/content/drive/MyDrive/gleason_project')

print(f'Project root: {PROJECT_ROOT}')

# Add project root to sys.path so all modules are importable
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## 1. SICAPv2 Dataset Check

The notebook expects the dataset at `<project_root>/SICAPv2/`.  
If not found, download from https://data.mendeley.com/datasets/9xxm58dvs3/ and extract there.

In [ ]:
SICAP_ROOT = PROJECT_ROOT / 'SICAPv2'

REQUIRED_PATHS = [
    SICAP_ROOT / 'images',
    SICAP_ROOT / 'masks',
    SICAP_ROOT / 'partition' / 'Test' / 'Test.xlsx',
    SICAP_ROOT / 'partition' / 'Test' / 'Train.xlsx',
    SICAP_ROOT / 'partition' / 'Validation' / 'Val1' / 'Train.xlsx',
    SICAP_ROOT / 'partition' / 'Validation' / 'Val1' / 'Test.xlsx',
]

if not SICAP_ROOT.exists():
    raise FileNotFoundError(
        f'SICAPv2 not found at {SICAP_ROOT}.\n'
        'Download from: https://data.mendeley.com/datasets/9xxm58dvs3/\n'
        'and extract so the folder structure is:\n'
        '  <project_root>/SICAPv2/images/   <-- .jpg patches\n'
        '  <project_root>/SICAPv2/partition/'
    )

missing = [p for p in REQUIRED_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError('Missing SICAPv2 files:\n' + '\n'.join(f'  {p}' for p in missing))

n_images = len(list((SICAP_ROOT / 'images').glob('*.jpg')))
print(f'SICAPv2 OK — {n_images} patches found at {SICAP_ROOT}')

## 2. Configuration

In [ ]:
import yaml

CONFIG_PATH = PROJECT_ROOT / 'training' / 'config.yaml'
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

# Optionally enable Drive checkpointing when running in Colab
cfg['save_to_drive'] = IN_COLAB

print('Config loaded:')
for k, v in cfg.items():
    if not isinstance(v, dict):
        print(f'  {k}: {v}')

## 3. Dataset & DataLoaders

In [ ]:
import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

from data.dataset import SICAPv2Dataset, CLASS_NAMES
from data.transforms import get_train_transforms, get_val_transforms

train_ds = SICAPv2Dataset(
    xlsx_path=PROJECT_ROOT / cfg['train_xlsx'],
    images_dir=PROJECT_ROOT / cfg['images_dir'],
    transform=get_train_transforms(cfg['input_size']),
)
val_ds = SICAPv2Dataset(
    xlsx_path=PROJECT_ROOT / cfg['val_xlsx'],
    images_dir=PROJECT_ROOT / cfg['images_dir'],
    transform=get_val_transforms(cfg['input_size']),
)
test_ds = SICAPv2Dataset(
    xlsx_path=PROJECT_ROOT / cfg['test_xlsx'],
    images_dir=PROJECT_ROOT / cfg['images_dir'],
    transform=get_val_transforms(cfg['input_size']),
)

class_weights = train_ds.get_class_weights()
sample_weights = [class_weights[lbl].item() for lbl in train_ds.labels]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')
print(f'Classes: {CLASS_NAMES}')
print(f'Class weights: {dict(zip(CLASS_NAMES, class_weights.tolist()))}')

In [ ]:
# Quick sanity: display one patch per class
import matplotlib.pyplot as plt
import numpy as np
from data.transforms import IMAGENET_MEAN, IMAGENET_STD

def denorm(t):
    img = t.numpy().transpose(1, 2, 0)
    return np.clip(img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)

seen = {}
for img, lbl in train_ds:
    if lbl not in seen:
        seen[lbl] = img
    if len(seen) == 4:
        break

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (lbl, img) in zip(axes, sorted(seen.items())):
    ax.imshow(denorm(img))
    ax.set_title(CLASS_NAMES[lbl])
    ax.axis('off')
plt.suptitle('One patch per class (training set)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Models

In [ ]:
from models.load_models import create_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

models = {}
for key, model_cfg in cfg['models'].items():
    m = create_model(model_cfg['timm_name'], num_classes=cfg['num_classes'])
    models[key] = m.to(device)
    params = sum(p.numel() for p in m.parameters())
    print(f'  {key}: {params/1e6:.1f}M params')

## 5. Training

Two-phase strategy:
- **Phase 1 (epochs 1–5):** freeze backbone, train head only
- **Phase 2 (epochs 6–20):** unfreeze full model, fine-tune end-to-end

In [ ]:
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast

from models.classifier_heads import freeze_backbone, unfreeze_all
from training.utils import AverageMeter, save_checkpoint, copy_checkpoint_to_drive, set_seed

set_seed(cfg['seed'])

class_weights_device = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights_device)

all_history = {}

for model_key, model in models.items():
    print(f'\n{"="*60}')
    print(f'Training: {model_key}')
    print(f'{"="*60}')

    freeze_backbone(model)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])
    scaler = GradScaler() if cfg['mixed_precision'] and device.type == 'cuda' else None

    ckpt_dir = PROJECT_ROOT / cfg['checkpoint_dir'] / model_key
    best_val_acc = 0.0
    history = []

    for epoch in range(1, cfg['epochs'] + 1):
        if epoch == cfg['unfreeze_epoch']:
            print(f'  Epoch {epoch}: Unfreezing full model.')
            unfreeze_all(model)
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=cfg['learning_rate'], weight_decay=cfg['weight_decay'],
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, T_max=cfg['epochs'] - epoch + 1,
            )

        # --- train ---
        model.train()
        tloss, tcorrect, ttotal = AverageMeter(), 0, 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            with autocast(enabled=scaler is not None):
                out = model(imgs)
                loss = criterion(out, lbls)
            if scaler:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss.backward(); optimizer.step()
            tloss.update(loss.item(), lbls.size(0))
            tcorrect += (out.argmax(1) == lbls).sum().item()
            ttotal += lbls.size(0)

        # --- val ---
        model.eval()
        vloss, vcorrect, vtotal = AverageMeter(), 0, 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                loss = criterion(out, lbls)
                vloss.update(loss.item(), lbls.size(0))
                vcorrect += (out.argmax(1) == lbls).sum().item()
                vtotal += lbls.size(0)

        scheduler.step()

        train_acc = tcorrect / ttotal
        val_acc   = vcorrect / vtotal
        is_best = val_acc > best_val_acc
        if is_best:
            best_val_acc = val_acc

        ckpt_name = f'epoch_{epoch:02d}.pt'
        state = dict(epoch=epoch, model_key=model_key,
                     state_dict=model.state_dict(),
                     val_acc=val_acc, optimizer=optimizer.state_dict())
        save_checkpoint(state, ckpt_dir, ckpt_name, is_best=is_best)

        if cfg.get('save_to_drive') and IN_COLAB:
            copy_checkpoint_to_drive(
                ckpt_dir / ckpt_name,
                Path(cfg['drive_checkpoint_dir']) / model_key,
            )

        row = dict(epoch=epoch,
                   train_loss=round(tloss.avg,4), train_acc=round(train_acc,4),
                   val_loss=round(vloss.avg,4),   val_acc=round(val_acc,4))
        history.append(row)
        marker = ' *' if is_best else ''
        print(f'  [{epoch:02d}/{cfg["epochs"]}] '
              f'train {train_acc:.4f} | val {val_acc:.4f}{marker}')

    all_history[model_key] = history
    print(f'  Best val acc: {best_val_acc:.4f}')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 4), sharey=True)
for ax, (key, hist) in zip(axes, all_history.items()):
    epochs = [r['epoch'] for r in hist]
    ax.plot(epochs, [r['train_acc'] for r in hist], label='train')
    ax.plot(epochs, [r['val_acc']   for r in hist], label='val')
    ax.axvline(cfg['freeze_epochs'], color='gray', linestyle='--', linewidth=0.8, label='unfreeze')
    ax.set_title(key)
    ax.set_xlabel('Epoch')
    ax.legend()
axes[0].set_ylabel('Accuracy')
plt.suptitle('Training curves', fontsize=13)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / cfg['figures_dir'] / 'training_curves.png', dpi=150)
plt.show()

## 6. Evaluation on Test Set

In [ ]:
from evaluation.metrics import compute_accuracy, evaluate_model

test_results = {}
for key, model in models.items():
    acc, per_class = compute_accuracy(model, test_loader, device)
    num_params = sum(p.numel() for p in model.parameters())
    test_results[key] = {
        'test_accuracy': round(acc, 4),
        'per_class_accuracy': per_class,
        'num_params': num_params,
    }
    print(f'{key}: test_acc={acc:.4f}  params={num_params/1e6:.1f}M')

## 7. Inference Speed Benchmark

In [ ]:
import time

WARMUP, TIMED = 20, 100
dummy = torch.randn(1, 3, cfg['input_size'], cfg['input_size'], device=device)

for key, model in models.items():
    model.eval()
    with torch.no_grad():
        for _ in range(WARMUP):
            model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(TIMED):
            model(dummy)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / TIMED * 1000
    test_results[key]['inference_ms_per_image'] = round(ms, 3)
    print(f'{key}: {ms:.2f} ms/image')

## 8. Results Summary Table

In [ ]:
import json

print(f'{"Model":<15} {"Test Acc":>10} {"Params (M)":>12} {"ms/image":>10}')
print('-' * 52)
for key, r in test_results.items():
    print(f'{key:<15} {r["test_accuracy"]:>10.4f} '
          f'{r["num_params"]/1e6:>12.1f} '
          f'{r.get("inference_ms_per_image", "N/A"):>10}')

# Save
metrics_path = PROJECT_ROOT / cfg['metrics_file']
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, 'w') as f:
    json.dump(test_results, f, indent=2)
print(f'\nMetrics saved to {metrics_path}')

In [ ]:
# Performance vs. Efficiency scatter plot
fig, ax = plt.subplots(figsize=(7, 5))
colors = {'resnet50': '#e15759', 'tiny_vit_11m': '#4e79a7', 'fastvit_t8': '#59a14f'}
for key, r in test_results.items():
    ax.scatter(
        r['num_params'] / 1e6,
        r['test_accuracy'],
        s=r.get('inference_ms_per_image', 10) * 15,
        color=colors.get(key, 'gray'),
        alpha=0.85,
        label=key,
        zorder=3,
    )
    ax.annotate(key, (r['num_params'] / 1e6, r['test_accuracy']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Test Accuracy')
ax.set_title('Performance vs. Efficiency (bubble size = inference time)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / cfg['figures_dir'] / 'pareto_tradeoff.png', dpi=150)
plt.show()

## 9. Grad-CAM Visualizations

Representative patches (2 per class × 4 classes) across all three models.

In [ ]:
from evaluation.gradcam import GradCAM, generate_gradcam_grid, _select_representative_patches

figures_dir = PROJECT_ROOT / cfg['figures_dir']
sample_indices = _select_representative_patches(test_ds, samples_per_class=2)

for key, model in models.items():
    print(f'Grad-CAM: {key}')
    generate_gradcam_grid(model, key, test_ds, device, figures_dir, sample_indices)

print('\nAll Grad-CAM figures saved to', figures_dir)

In [ ]:
# Display saved Grad-CAM figures inline
from IPython.display import Image as IPImage, display
for key in models:
    fig_path = figures_dir / f'gradcam_{key}.png'
    if fig_path.exists():
        print(f'\n{key}')
        display(IPImage(str(fig_path)))

## 10. Done

All outputs saved to `results/`:
- `results/metrics.json` — accuracy, inference time, parameter count
- `results/checkpoints/<model>/best_*.pt` — best checkpoints
- `results/figures/training_curves.png`
- `results/figures/pareto_tradeoff.png`
- `results/figures/gradcam_<model>.png` (one per model)

---
**LLM Disclosure:** [List here which parts of this code/notebook were generated or significantly assisted by an LLM, per course policy.]